In [1]:
import numpy as np
import pickle
import os
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# ---------------------------------------------------------
# [Step 1] 데이터 로딩
# ---------------------------------------------------------
print("== 1. Loading data ==")
data_dir = './cifar-10-batches-py/' 

# CIFAR-10 데이터 불러오는 함수임.
def load_cifar_batch(file_path):
    with open(file_path, 'rb') as f:
        d = pickle.load(f, encoding='bytes') 
        data_matrix = d[b'data']
        labels_list = d[b'labels']
    return data_matrix, labels_list

# 훈련 데이터 5만 장 합치기
trn_imgs = []
trn_lbls = []
for i in range(1, 6):
    b_path = os.path.join(data_dir, f'data_batch_{i}')
    x_tmp, y_tmp = load_cifar_batch(b_path)
    trn_imgs.append(x_tmp)
    trn_lbls.append(y_tmp)

trn_imgs = np.concatenate(trn_imgs)
trn_lbls = np.concatenate(trn_lbls)

# 테스트 데이터 1만 장 불러오기
tst_imgs, tst_lbls = load_cifar_batch(os.path.join(data_dir, 'test_batch'))
tst_lbls = np.array(tst_lbls)

print(f"Training data shape: {trn_imgs.shape}")
print(f"Test data shape: {tst_imgs.shape}")


# ---------------------------------------------------------
# [Step 2] 데이터 전처리 (정규화)
# ---------------------------------------------------------
print("\n== 2. Normalizing data ==")
# 평균값을 빼주어 데이터 스케일을 맞추는 과정임.
mean_img = np.mean(trn_imgs, axis=0)
trn_x_norm = trn_imgs.astype('float32') - mean_img
tst_x_norm = tst_imgs.astype('float32') - mean_img
print("Normalization done")


# ---------------------------------------------------------
# [Step 3] 선형 분류기 학습 (단 1회)
# ---------------------------------------------------------
print("\n== 3. Training Linear Classifier ==")
# 5-Fold 반복 다 빼고 가장 무난한 파라미터(C=1e-4)로 1번만 후딱 학습시킴.
svm_model = LinearSVC(loss='hinge', C=1e-4, max_iter=1000, random_state=42)

# 전체 데이터 5만 장 투입! (단일 학습이라 10~20초면 끝남)
svm_model.fit(trn_x_norm, trn_lbls)
print("Training finished!")


# ---------------------------------------------------------
# [Step 4] 최종 평가 및 지표 계산
# ---------------------------------------------------------
print("\n== 4. Evaluating Final Test Set ==")
# 테스트 데이터 예측
final_preds = svm_model.predict(tst_x_norm)

# 4대 지표 계산
acc = accuracy_score(tst_lbls, final_preds)
prec = precision_score(tst_lbls, final_preds, average='macro', zero_division=0)
rec = recall_score(tst_lbls, final_preds, average='macro', zero_division=0)
f1 = f1_score(tst_lbls, final_preds, average='macro', zero_division=0)

# 결과 출력
print(f"Accuracy : {acc*100:.2f}%")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-Score : {f1:.4f}")

== 1. Loading data ==
Training data shape: (50000, 3072)
Test data shape: (10000, 3072)

== 2. Normalizing data ==
Normalization done

== 3. Training Linear Classifier ==
Training finished!

== 4. Evaluating Final Test Set ==
Accuracy : 24.35%
Precision: 0.2379
Recall   : 0.2435
F1-Score : 0.2212


C:\Users\jason\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
